### This table displays the dataset with newly calculated daily laying rates and ISO week numbers for each record.

In [1]:
import pandas as pd
from pathlib import Path

# Load clean dataset (using workspace root path)
clean_path = Path("data/clean/eggs_clean.csv")
df = pd.read_csv(clean_path, parse_dates=["date"])

# 1. Add rate (eggs ÷ birds) and week (ISO week number)
df["rate"] = df["eggs"] / df["birds"]
df["week"] = df["date"].dt.isocalendar().week

# --- Golden Rule / Checks ---
print(f"Total Rows: {len(df)}")
print(f"Rate Min: {df['rate'].min():.3f}")
print(f"Rate Max: {df['rate'].max():.3f}")

print("\n--- Week Value Counts ---")
print(df["week"].value_counts().sort_index())

df.head()

Total Rows: 32
Rate Min: 0.676
Rate Max: 0.842

--- Week Value Counts ---
week
2     2
7     2
11    1
15    2
19    1
24    2
28    2
33    2
37    2
40    3
41    7
42    4
46    1
50    1
Name: count, dtype: Int64


,date,flock,eggs,birds,recorded_by,rate,week
0,2026-01-10,A,970,1200,Wanjiku,0.808333,2
1,2026-10-01,B,641,800,Otieno,0.801250,40
2,2026-01-10,C,432,600,Akinyi,0.720000,2
3,2026-02-10,A,1007,1200,Wanjiku,0.839167,7
4,2026-10-02,B,617,800,Otieno,0.771250,40


### This table shows the overall production summary, total bird-days, and aggregate laying rate for each flock across the entire period.

In [2]:
# Group by flock to aggregate total eggs, total bird-days (sum of birds), and days (row count)
flock_summary = df.groupby("flock").agg(
    eggs=("eggs", "sum"),
    bird_days=("birds", "sum"),
    days=("date", "count")
).reset_index()

# Calculate overall rate: total eggs ÷ total bird-days
flock_summary["rate"] = flock_summary["eggs"] / flock_summary["bird_days"]

# Format rate as percentage to one decimal place for easy visual comparison
flock_summary["rate_display"] = (flock_summary["rate"] * 100).round(1).astype(str) + "%"

# Display summary table
flock_summary[["flock", "eggs", "bird_days", "days", "rate_display"]]

,flock,eggs,bird_days,days,rate_display
0,A,9832,11993,10,82.0%
1,B,7704,9600,12,80.2%
2,C,4261,5994,10,71.1%


In [ ]:
### This table shows weekly production metrics—including active days, total eggs, and aggregate laying rate—broken down by individual flock week by week.

In [3]:
# Group by flock and week
weekly_summary = df.groupby(["flock", "week"]).agg(
    start_date=("date", "min"),
    end_date=("date", "max"),
    days=("date", "count"),
    eggs=("eggs", "sum"),
    bird_days=("birds", "sum")
).reset_index()

# Calculate weekly aggregate rate
weekly_summary["rate"] = weekly_summary["eggs"] / weekly_summary["bird_days"]
weekly_summary["rate_display"] = (weekly_summary["rate"] * 100).round(1).astype(str) + "%"

# Format date range for readability
weekly_summary["dates"] = (
    weekly_summary["start_date"].dt.strftime("%d %b") + " – " + 
    weekly_summary["end_date"].dt.strftime("%d %b")
)

# Select and reorder columns to match inspection goals
weekly_summary[[
    "flock", "week", "dates", "days", "eggs", "rate_display"
]]

,flock,week,dates,days,eggs,rate_display
0,A,2,10 Jan – 10 Jan,1,970,80.8%
1,A,7,10 Feb – 10 Feb,1,1007,83.9%
2,A,15,10 Apr – 10 Apr,1,959,79.9%
3,A,19,10 May – 10 May,1,996,83.0%
4,A,24,10 Jun – 10 Jun,1,972,81.1%
5,A,28,10 Jul – 10 Jul,1,1009,84.2%
6,A,33,10 Aug – 10 Aug,1,1010,84.2%
7,A,37,10 Sep – 10 Sep,1,961,80.2%
8,A,41,10 Oct – 10 Oct,1,998,83.2%
9,A,50,10 Dec – 10 Dec,1,950,79.3%


### Method 1: The mean of the daily rate column across all records.

In [4]:
# Method 1: Mean of the daily rate column
method_1_rate = df["rate"].mean()
print(f"Method 1 (Mean of daily rates): {method_1_rate * 100:.1f}%")

Method 1 (Mean of daily rates): 77.9%


### Method 2: The mean of the three individual flock rates from Step 2.

In [5]:
# Method 2: Mean of the three flock rates
method_2_rate = flock_summary["rate"].mean()
print(f"Method 2 (Mean of flock rates): {method_2_rate * 100:.1f}%")

Method 2 (Mean of flock rates): 77.8%


### Method 3: Total farm eggs divided by total farm bird-days (correct weighted aggregation).

In [6]:
# Method 3: Total eggs ÷ total bird-days for the entire farm
total_farm_eggs = df["eggs"].sum()
total_farm_bird_days = df["birds"].sum()
method_3_rate = total_farm_eggs / total_farm_bird_days

print(f"Method 3 (Total eggs ÷ Total bird-days): {method_3_rate * 100:.1f}%")

Method 3 (Total eggs ÷ Total bird-days): 79.0%


Only the third method is correct because it weights each day and flock by its actual population size rather than treating every observation or flock equally. The first two methods give too much weight to Flock C, whose smaller bird population and lower performance disproportionately drag down the unweighted average.

### This table highlights the top three highest-performing days by laying rate for each individual flock.

In [7]:
# Sort by flock (ascending) and rate (descending), then group and take top 3 per flock
top_days = (
    df.sort_values(by=["flock", "rate"], ascending=[True, False])
    .groupby("flock")
    .head(3)
    .copy()
)

# Format rate as percentage to one decimal place
top_days["rate_display"] = (top_days["rate"] * 100).round(1).astype(str) + "%"

# Format date for readability (e.g., "8 Oct")
top_days["date_formatted"] = top_days["date"].dt.strftime("%d %b")

# Select and display final columns
top_days[[
    "flock", "date_formatted", "eggs", "birds", "rate_display"
]].reset_index(drop=True)

,flock,date_formatted,eggs,birds,rate_display
0,A,10 Aug,1010,1199,84.2%
1,A,10 Jul,1009,1199,84.2%
2,A,10 Feb,1007,1200,83.9%
3,B,10 Oct,669,800,83.6%
4,B,05 Oct,667,800,83.4%
5,B,08 Oct,656,800,82.0%
6,C,10 Aug,447,599,74.6%
7,C,10 Mar,445,600,74.2%
8,C,10 Nov,436,599,72.8%


In [ ]:
### This table breaks down each flock's total egg production into full trays (30 eggs/tray), loose eggs, and total calculated revenue at 450 KES per tray based on verified records.

In [8]:
import numpy as np

# Create a copy of flock_summary to avoid modifying the original dataframe
revenue_df = flock_summary[["flock", "eggs"]].copy()

# Calculate trays (30 eggs per tray) and loose eggs
revenue_df["trays"] = revenue_df["eggs"] // 30
revenue_df["loose_eggs"] = revenue_df["eggs"] % 30

# Calculate revenue at 450 KES per tray
revenue_df["revenue_kes"] = revenue_df["trays"] * 450

# Format revenue for display with commas
revenue_df["revenue_display"] = revenue_df["revenue_kes"].apply(lambda x: f"{x:,.0f}")

# Calculate farm totals
total_eggs = revenue_df["eggs"].sum()
total_trays = revenue_df["trays"].sum()
total_loose = revenue_df["loose_eggs"].sum()
farm_revenue = revenue_df["revenue_kes"].sum()

# Append a Farm Total row
total_row = pd.DataFrame({
    "flock": ["Farm Total"],
    "eggs": [total_eggs],
    "trays": [total_trays],
    "loose_eggs": [total_loose],
    "revenue_kes": [farm_revenue],
    "revenue_display": [f"{farm_revenue:,.0f}"]
})

revenue_table = pd.concat([revenue_df, total_row], ignore_index=True)

# Select and display final columns
revenue_table[[
    "flock", "trays", "loose_eggs", "revenue_display"
]]

,flock,trays,loose_eggs,revenue_display
0,A,327,22,"147,150"
1,B,256,24,"115,200"
2,C,142,1,"63,900"
3,Farm Total,725,47,"326,250"


### This section categorizes flock performance into 'good', 'watch', or 'act now' based on laying rate thresholds, and filters weekly records for targeted intervention.

In [9]:
# Define classification function based on rate thresholds
def classify_performance(rate):
    if rate >= 0.80:
        return "good"
    elif rate >= 0.75:
        return "watch"
    else:
        return "act now"

# 1. Apply classification to overall flock summary (from Step 2)
flock_summary["status"] = flock_summary["rate"].apply(classify_performance)

print("--- Overall Flock Performance Status ---")
display(flock_summary[["flock", "eggs", "bird_days", "rate_display", "status"]])

# 2. Apply classification to weekly summary (from Step 3)
weekly_summary["status"] = weekly_summary["rate"].apply(classify_performance)

# Filter to show only weeks that require attention (below 'good', i.e., 'watch' or 'act now')
needs_attention = weekly_summary[weekly_summary["status"] != "good"].copy()

print("\n--- Flock-Weeks Requiring Attention (Watch or Act Now) ---")
display(needs_attention[[
    "flock", "week", "dates", "days", "eggs", "rate_display", "status"
]].reset_index(drop=True))

--- Overall Flock Performance Status ---


,flock,eggs,bird_days,rate_display,status
0,A,9832,11993,82.0%,good
1,B,7704,9600,80.2%,good
2,C,4261,5994,71.1%,act now



--- Flock-Weeks Requiring Attention (Watch or Act Now) ---


,flock,week,dates,days,eggs,rate_display,status
0,A,15,10 Apr – 10 Apr,1,959,79.9%,watch
1,A,50,10 Dec – 10 Dec,1,950,79.3%,watch
2,B,40,01 Oct – 03 Oct,3,1912,79.7%,watch
3,B,42,12 Oct – 16 Oct,4,2512,78.5%,watch
4,C,2,10 Jan – 10 Jan,1,432,72.0%,act now
5,C,7,10 Feb – 10 Feb,1,408,68.0%,act now
6,C,11,10 Mar – 10 Mar,1,445,74.2%,act now
7,C,15,10 Apr – 10 Apr,1,421,70.2%,act now
8,C,24,10 Jun – 10 Jun,1,434,72.5%,act now
9,C,28,10 Jul – 10 Jul,1,410,68.4%,act now
